# Lun 07/12 · 11:30 — TP v1.0 : tests avec pytest

Lun 07/12, TP de 12:00 à 12:50. Correction à 12:50, TP fini ou non, puis bilan et rappel de la soutenance du jeu 10/12. La référence v1.0 est publiée à 13:00.

## Objectif

`fleetcheck` reçoit ses tests, qui tournent sans réseau ni serveur :

- les règles pures d'abord : `exit_code`, `global_status`, `validate` ;
- `load` sur des fichiers écrits dans un dossier temporaire ;
- `check_health` et `check_service` avec une fausse session `requests` et un `ping` remplacé.

Un test n'a de valeur que s'il échoue quand le code est faux : à chaque étape, le notebook rejoue vos tests sur une copie volontairement cassée du package.

## Mode d'emploi du notebook

- Copiez ce notebook à la racine de votre dépôt, à côté du dossier `fleetcheck/`. Les tests s'écrivent dans `tests/test_checks.py` et `tests/test_inventory.py`, ouverts dans JupyterLab ; le notebook lance `python -m pytest` en sous-processus et lit son résumé.
- **Arrêtez le serveur simulé** (Ctrl+C dans son terminal) : les tests ne doivent pas en dépendre.
- Exécutez d'abord la cellule d'outils ci-dessous. Chaque vérification affiche `OK`, ou `KO` avec ce qui était attendu.
- La cellule d'outils ajoute `pytest_run(...)`, qui lance `python -m pytest` et renvoie le résultat avec le nombre de tests passés (`r.passes`), et `casse(module, code, ...)`, qui copie `fleetcheck/` et `tests/` dans un dossier temporaire, ajoute `code` à la fin de `fleetcheck/<module>.py` (la fonction du même nom est remplacée) et y relance vos tests. Ne modifiez pas la cellule « Copies cassées ».
- **Prédire** et **Justifier** marquent les moments clés du TP. Répondez dans la cellule `# Ma réponse :` qui suit, **avant** d'exécuter la suite.
- Rendu en fin de TP : passez `version` à `"1.0.0"` dans `pyproject.toml` et réinstallez (section « Release v1.0 »), menu *Kernel > Restart Kernel and Run All Cells*, puis commit et push de `tests/`, `requirements.txt`, `pyproject.toml` et de ce notebook.

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args):
    """Lance fleetcheck.py comme dans un terminal (vu en v0.8 : subprocess)."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r


def run_module(*args):
    """Lance le package comme dans un terminal : python -m fleetcheck ..."""
    r = subprocess.run([sys.executable, "-m", "fleetcheck", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r


def pytest_run(*args, cwd="."):
    """Lance python -m pytest ; r.passes = nombre de tests passés."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", *args], cwd=cwd,
                       capture_output=True, encoding="utf-8",
                       env={**os.environ, "PYTHONUTF8": "1", "PYTHONPATH": str(Path(cwd).resolve())})
    resume = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr.strip()
    passes = re.search(r"(\d+) passed", resume)
    r.passes = int(passes[1]) if passes else 0
    print(resume)
    print(f"[code de sortie : {r.returncode}]")
    return r


def casse(module, code, *args):
    """Rejoue les tests sur une copie où code est ajouté à la fin de fleetcheck/<module>.py."""
    with tempfile.TemporaryDirectory() as copie:
        shutil.copytree("fleetcheck", Path(copie) / "fleetcheck", ignore=shutil.ignore_patterns("__pycache__"))
        shutil.copytree("tests", Path(copie) / "tests", ignore=shutil.ignore_patterns("__pycache__"))
        cible = Path(copie) / "fleetcheck" / f"{module}.py"
        cible.write_text(cible.read_text(encoding="utf-8") + "\n\n" + code, encoding="utf-8")
        print(f"copie cassée ({module}) :", end=" ")
        return pytest_run(*args, cwd=copie)

## Point de départ

Votre dépôt de v0.12 : `fleetcheck/`, `pyproject.toml`, `config.json`, `requirements.txt`.

En retard ou en panne : copiez dans votre dépôt les fichiers de `reference/` au tag `v0.12` du dépôt public.

Vérifiez avant de commencer :

In [ ]:
r = run_module("--help")
check(("--config" in r.stdout, r.returncode), (True, 0))

## Copies cassées

Les versions fausses que `casse(...)` injecte, une par étape. Cellule fournie, à exécuter sans la modifier.

In [ ]:
EXIT_CODE_FAUX = '''
def exit_code(statuses, invalid_count):
    return 0
'''

VALIDATE_FAUX = '''
_validate_vrai = validate


def validate(fields):
    if len(fields) == 5 and fields[3].strip() == "70000":
        server = _validate_vrai([*fields[:3], "8001", fields[4]])
        server["port"] = 70000
        return server
    return _validate_vrai(fields)
'''

RETRY_4XX_FAUX = '''
_check_health_vrai = check_health


def check_health(session, url, config):
    result = _check_health_vrai(session, url, config)
    essais = result["attempts"]
    while result["http_status"] is not None and 400 <= result["http_status"] < 500 and essais < config["attempts"]:
        result = _check_health_vrai(session, url, config)
        essais += 1
    return {**result, "attempts": essais}
'''

## Étapes

| Fin | Étape |
|---|---|
| 12:15 | 1. pytest, `test_exit_code` paramétré et `test_global_status_is_worst` |
| 12:35 | 2. Tests de l'inventaire : `validate`, `pytest.raises`, `load` sur `tmp_path` |
| 12:50 | 3. Tests HTTP sans réseau : `Mock` et `monkeypatch` |

À 12:25, si plus d'un tiers de la salle n'a pas `test_exit_code` vert, l'enseignant l'écrit au tableau ; l'étape 3 se limite à `test_health_retries_until_success`. Dans ce mode, deux `KO` de l'étape 3 sont attendus : « au moins 18 tests » et la copie qui réessaie les `4xx`. Les tests manquants se terminent en devoir, et ces deux vérifications repassent alors au vert.

### 1. pytest et les règles pures — fin 12:15

- Installez pytest dans le venv du noyau, et ajoutez la ligne `pytest==9.1.1` à `requirements.txt` (un outil de développement : il ne va pas dans les `dependencies` de `pyproject.toml`).
- Créez le dossier `tests/` et `tests/test_checks.py`, qui commence par `from fleetcheck import checks`.
- `test_exit_code`, décoré par `@pytest.mark.parametrize("statuses, invalid, expected", [...])`, couvre les 5 cas de la règle : `(["ok", "ok"], 0, 0)`, `(["ok", "degraded"], 0, 0)`, `(["ok", "down"], 0, 1)`, `(["ok", "ok"], 1, 2)`, `(["down"], 1, 2)`. Corps : `assert checks.exit_code(statuses, invalid) == expected`.
- `test_global_status_is_worst` : le pire statut l'emporte, et une liste vide donne `"ok"`.

In [ ]:
%pip install pytest==9.1.1

**Justifier** — Pourquoi vérifier que le test échoue sur une copie où `exit_code` renvoie toujours `0` ?

In [ ]:
# Ma réponse :

Vérification : au moins 6 tests verts, la ligne `pytest==9.1.1` dans `requirements.txt`, et au moins un test rouge sur la copie cassée.

In [ ]:
r = pytest_run("tests/test_checks.py")
check((r.returncode, r.passes >= 6), (0, True))
check("pytest==9.1.1" in Path("requirements.txt").read_text(encoding="utf-8").split(), True)
r = casse("checks", EXIT_CODE_FAUX, "tests/test_checks.py")
check(r.returncode != 0, True)

### 2. Tests de l'inventaire — fin 12:35

`tests/test_inventory.py`, qui commence par `from fleetcheck.inventory import FatalError, InvalidServer, load, validate` :

- `validate(["web-1", "prod", "127.0.0.1", "8001", "web"])` renvoie le serveur avec le port en `int` ;
- un test paramétré : ces 5 lignes lèvent `InvalidServer`, vérifié par `with pytest.raises(InvalidServer, match=...)` sur un morceau du message :

  | Champs | Message attendu (extrait) |
  |---|---|
  | `["web-1", "prod", "127.0.0.1", "abc", "web"]` | `port non entier` |
  | `["web-1", "prod", "127.0.0.1", "70000", "web"]` | `port hors` |
  | `["web-1", "prod", "300.0.0.1", "8001", "web"]` | `IP invalide` |
  | `["web-1", "", "127.0.0.1", "8001", "web"]` | `'env' vide` |
  | `["web-1", "prod", "127.0.0.1"]` | `3 champs` |

- `load` sur un CSV de trois lignes écrit dans `tmp_path` (une valide, une au port `abc`) : un serveur valide, et une ligne invalide numérotée `3` ;
- `load` sur un CSV sans ligne valide lève `FatalError`.

Écrivez les fichiers de test avec `write_text(..., encoding="utf-8")`.

Vérification : au moins 8 tests verts, et au moins un test rouge sur une copie où `validate` accepte le port `70000`.

In [ ]:
r = pytest_run("tests/test_inventory.py")
check((r.returncode, r.passes >= 8), (0, True))
r = casse("inventory", VALIDATE_FAUX, "tests/test_inventory.py")
check(r.returncode != 0, True)

### 3. Tests HTTP sans réseau — fin 12:50

Dans `tests/test_checks.py` (imports : `from datetime import timedelta`, `from unittest.mock import Mock`, `import requests`) :

- une config de test avec `"attempts": 3` et `"backoff_s": 0`, et une petite fonction qui fabrique une réponse simulée : `Mock(status_code=status, elapsed=timedelta(seconds=0.01), json=Mock(return_value=corps))` ;
- `check_health` avec `Mock(get=Mock(side_effect=[r503, r503, r200]))` : `http_status` `200` et `attempts` `3` ;
- `check_health` avec une réponse `404` : `attempts` `1` et `session.get.call_count == 1` ;
- `check_health` avec `side_effect=requests.Timeout` : le résultat complet après 3 essais (`error` `"timeout"`) ;
- `check_service` avec `monkeypatch.setattr(checks, "ping", lambda ip, timeout_s: False)` : `down`, `health` à `None`, et `session.get.assert_not_called()`.

**Prédire** — Un test de `check_health` sans `Mock`, contre le serveur simulé, sur `api-stg-1` : combien de temps dure-t-il, et passe-t-il sur la machine de CI ?

In [ ]:
# Ma réponse :

**Justifier** — Pourquoi `backoff_s` vaut-il `0` dans la config des tests ?

In [ ]:
# Ma réponse :

Vérification : toute la suite, au moins 18 tests verts en moins de 10 s (démarrage de Python compris), serveur simulé arrêté ; au moins un test rouge sur une copie où `check_health` réessaie aussi les `4xx`.

In [ ]:
import time

debut = time.monotonic()
r = pytest_run()
check((r.returncode, r.passes >= 18, time.monotonic() - debut < 10), (0, True, True))
r = casse("checks", RETRY_4XX_FAUX)
check(r.returncode != 0, True)

## Critère final

`python -m pytest` est vert sans réseau ni serveur, et rouge sur chacune des trois copies cassées. Sur la référence v1.0 :

```text
.......................                                                  [100%]
23 passed in 0.09s
```

In [ ]:
r = pytest_run()
check(r.returncode, 0)
check([casse(m, c).returncode != 0 for m, c in
       [("checks", EXIT_CODE_FAUX), ("inventory", VALIDATE_FAUX), ("checks", RETRY_4XX_FAUX)]],
      [True, True, True])

## Release v1.0

Le geste de release : la version déclarée passe à `1.0.0`, et l'installation la rend visible. Dans `pyproject.toml`, `version = "1.0.0"`, puis :

In [ ]:
%pip install -e .

Vérification : la version est une métadonnée de l'installation, relue après la réinstallation.

In [ ]:
import importlib.metadata

check(importlib.metadata.version("fleetcheck"), "1.0.0")

## Pour aller plus loin

- `tests/test_report.py` : `archive_old` sur `tmp_path`, avec un rapport reculé de 10 jours par `os.utime` (TP v0.7) ; vérifiez le contenu du zip avec `zipfile.ZipFile(...).namelist()`.
- `python -m pytest --durations=3` : les trois tests les plus lents. Un test unitaire qui dépasse 0,1 s attend souvent un réseau ou un `time.sleep`.
- Écrivez votre propre copie cassée (`global_status` qui renvoie le premier statut) et vérifiez que vos tests la détectent.